# 📊 Customer Churn Prediction Pipeline

This notebook demonstrates an end-to-end Machine Learning pipeline to predict customer churn using the **IBM Telco Customer Churn** dataset. Customer churn occurs when customers stop doing business with a company. Predicting churn is critical for subscription-based businesses to retain customers.

## ⚙️ Pipeline Highlights:
1. **Exploratory Data Analysis (EDA)**: Visualizing distributions, churn rates, and feature correlations.
2. **Data Preprocessing & Scaling**: Clean missing values, encode categorical variables, and scale numerical columns.
3. **Model Selection**: Evaluate Logistic Regression, Random Forest, and XGBoost.
4. **Class Imbalance Management**: Apply class weighting and metric adjustments.
5. **Hyperparameter Tuning**: Perform GridSearchCV to optimize models.
6. **Evaluation**: Generate detailed reports, ROC-AUC, and feature importances.

### 1. Import Libraries & Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
print("Libraries successfully imported!")

### 2. Ingest Data

In [ ]:
# Load the dataset
data_path = os.path.join("..", "data", "raw", "customer_churn.csv")
if not os.path.exists(data_path):
    import sys
    sys.path.append(os.path.abspath("../src"))
    from data_loader import download_data
    DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Customer-Churn.csv"
    download_data(DATA_URL, data_path)

df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
df.head()

### 3. Exploratory Data Analysis (EDA)

In [ ]:
# 3.1 Churn Distribution (Class Imbalance)
plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Distribution of Customer Churn')
plt.xlabel('Churn Status')
plt.ylabel('Count')
plt.show()

churn_rate = df['Churn'].value_counts(normalize=True) * 100
print(f"No Churn: {churn_rate['No']:.2f}%")
print(f"Churn: {churn_rate['Yes']:.2f}%")

In [ ]:
# 3.2 Tenure vs Monthly Charges colored by Churn
plt.figure(figsize=(10, 6))
sns.scatterplot(x='tenure', y='MonthlyCharges', hue='Churn', data=df, alpha=0.5, palette='coolwarm')
plt.title('Tenure vs Monthly Charges')
plt.xlabel('Tenure (Months)')
plt.ylabel('Monthly Charges ($)')
plt.show()

In [ ]:
# 3.3 Churn Rate by Contract Type
plt.figure(figsize=(8, 5))
sns.countplot(x='Contract', hue='Churn', data=df, palette='viridis')
plt.title('Churn Rate by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Count')
plt.show()

### 4. Data Preprocessing & Pipeline Construction

In [ ]:
# Replace spaces with NaN in TotalCharges and cast to float
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# Median imputation for TotalCharges
total_charges_median = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(total_charges_median)

# Drop identifier column
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Split Column Types
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = [col for col in X.columns if col not in numerical_cols]

# Preprocessor Pipeline
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first', handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Train Test Split with Stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

### 5. Model Benchmarking

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train), random_state=42, eval_metric='logloss')
}

results = {}
for name, clf in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    auc_score = roc_auc_score(y_test, y_prob)
    results[name] = {
        'pipeline': pipeline,
        'auc': auc_score,
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    print(f"{name} ROC-AUC: {auc_score:.4f}")

### 6. Hyperparameter Tuning (Grid Search on Best Model)

In [ ]:
# Find best model based on AUC
best_model_name = max(results, key=lambda x: results[x]['auc'])
best_pipeline = results[best_model_name]['pipeline']
print(f"Optimizing {best_model_name}...")

if best_model_name == "Logistic Regression":
    param_grid = {'classifier__C': [0.01, 0.1, 1.0, 10.0]}
elif best_model_name == "Random Forest":
    param_grid = {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [10, 15, None],
        'classifier__min_samples_split': [2, 5, 10]
    }
else: # XGBoost
    param_grid = {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [3, 5, 7],
        'classifier__learning_rate': [0.01, 0.1, 0.2]
    }

grid_search = GridSearchCV(best_pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

tuned_pipeline = grid_search.best_estimator_
print(f"Tuned Parameters: {grid_search.best_params_}")

### 7. Evaluation & Visualizations

In [ ]:
y_pred_tuned = tuned_pipeline.predict(X_test)
y_prob_tuned = tuned_pipeline.predict_proba(X_test)[:, 1]

print("Tuned Model Classification Report:")
print(classification_report(y_test, y_pred_tuned))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_tuned)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ROC Curve comparison
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['auc']:.4f})")

fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_tuned)
plt.plot(fpr_t, tpr_t, label=f"Tuned {best_model_name} (AUC = {roc_auc_score(y_test, y_prob_tuned):.4f})", linestyle='--', color='black')

plt.plot([0, 1], [0, 1], linestyle='--', color='red')
plt.title('ROC Curves Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Save the pipeline
import joblib
os.makedirs("../models", exist_ok=True)
joblib.dump(tuned_pipeline, "../models/customer_churn_pipeline.joblib")
print("Tuned pipeline exported to models/customer_churn_pipeline.joblib!")